# EDA — Subastas Públicas de Inmuebles en España
Fuente: Portal de Subastas del BOE (subastas.boe.es)<br>
Período analizado: Diciembre 2025

---
## Hipótesis
> *"La mayoría de los inmuebles subastados en el Portal del BOE 
> durante diciembre 2025 se ofertan por debajo del 50% de su valor 
> de tasación, lo que representa una oportunidad significativa de 
> compra para inversores y particulares"*

## Objetivo
Analizar el comportamiento de las subastas públicas de inmuebles 
en España: precios, descuentos sobre tasación, organismos 
convocantes y distribución geográfica.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('./utils')
from funciones import limpiar_dataframe, calcular_pct, calcular_duracion_dias  ,filtrar_tributarios, resumen_dataset

# Configuración de visualizaciones
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')


# **1. VISUALIZACION**

### **1.1 Carga de los 3 dataset de subastas (Octubre, noviembre y diciembre) que está en la ruta './data/'**

In [2]:
df_oct = pd.read_csv('./data/subastas_inmuebles_oct2025.csv')
df_nov = pd.read_csv('./data/subastas_inmuebles_nov2025.csv')
df_dic = pd.read_csv('./data/subastas_inmuebles_dic2025.csv')

print(f'Octubre   : {len(df_oct):,} registros')
print(f'Noviembre : {len(df_nov):,} registros')
print(f'Diciembre : {len(df_dic):,} registros')
print(f'Total     : {len(df_oct) + len(df_nov) + len(df_dic):,} registros')

Octubre   : 1,671 registros
Noviembre : 1,518 registros
Diciembre : 1,537 registros
Total     : 4,726 registros


### **1.2 Concatenar los 3 dataset**

In [3]:
df = pd.concat([df_oct, df_nov, df_dic], ignore_index=True)

print(f'Dataset combinado: {df.shape}')
print(f'\n¿Hay duplicados?')
print(f'Filas duplicadas: {df.duplicated(subset="referencia").sum()}')

Dataset combinado: (4726, 23)

¿Hay duplicados?
Filas duplicadas: 0


In [4]:
#Primera 5 filas
df.head()

,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,fecha_conclusion,cantidad_reclamada_eur,lotes,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,forma_adjudicacion
0,SUB-JA-2024-231148,1,JUZGADO 1 INSTANCIA 2 - LEON,0146/21,Pendiente de finalización y devolución de depó...,01/10/2025 a las 18:00:00,FINCA NÚMERO DIEZ.- Vivienda en la planta cuar...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2024-231148,SUB-JA-2024-231148,...,2025-10-01,38074.96,1,BOE-B-2025-31964,110887.98,0.0,NaN,NaN,5544.40,NaN
1,SUB-JA-2025-237883,1,UNIDAD SUBASTAS JUDICIALES MURCIA - MURCIA,0353/17,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,Id. lote. 237883L1. Finca registral 37881. Pis...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-237883,SUB-JA-2025-237883,...,2025-10-01,56189.91,1,BOE-B-2025-31986,107904.68,0.0,NaN,2158.1,5395.23,NaN
2,SUB-JA-2025-242137,4,JUZGADO 1 INST E INSTRUCC. 1 - SOLSONA,0006/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,"Subasta con varios lotes. Lote 1: URBANA, Parc...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-242137,SUB-JA-2025-242137,...,2025-10-01,112641.00,4,BOE-B-2025-31977,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
3,SUB-JA-2025-244592,2,JUZGADO 1 INST E INSTRUCC. 1 - GANDESA,0442/20,Cancelada,NaN,Subasta con varios lotes. Lote 1: 1. URBANA. C...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-244592,SUB-JA-2025-244592,...,NaN,205737.80,2,BOE-B-2025-31954,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
4,SUB-JA-2025-245672,1,Sección Civil e Instrucción TI Blanes. Plz.n 1...,0719/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,URBANA: FINCA ESPECIAL. ENTIDAD NUMERO TRES.- ...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-245672,SUB-JA-2025-245672,...,2025-10-01,179629.82,1,BOE-B-2025-31945,259600.00,0.0,NaN,5192.0,12980.00,NaN


In [5]:
#Tipos de datos
df.info() #=====================aqui me quede

<class 'pandas.DataFrame'>
RangeIndex: 4726 entries, 0 to 4725
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   referencia                4726 non-null   str    
 1   num_lotes_listado         4726 non-null   int64  
 2   organismo                 4726 non-null   str    
 3   expediente                3360 non-null   str    
 4   estado                    4726 non-null   str    
 5   fecha_conclusion_listado  4364 non-null   str    
 6   descripcion               4653 non-null   str    
 7   url_detalle               4726 non-null   str    
 8   id_sub                    4726 non-null   str    
 9   referencia_det            4726 non-null   str    
 10  tipo_subasta              4726 non-null   str    
 11  cuenta_expediente         3360 non-null   str    
 12  fecha_inicio              4364 non-null   str    
 13  fecha_conclusion          4364 non-null   str    
 14  cantidad_reclamada_

In [6]:
#Estadísticas básicas
df.describe()

,num_lotes_listado,cantidad_reclamada_eur,lotes,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur
count,4726.000000,3.675000e+03,4726.000000,4.062000e+03,4.081000e+03,1.321000e+03,3465.000000,4.081000e+03
mean,1.826280,4.816601e+05,1.826280,2.527936e+05,1.042575e+05,5.205519e+04,4631.335076,1.371823e+04
std,4.901061,7.098250e+06,4.901061,9.245536e+05,6.142216e+05,3.647587e+05,18454.308898,4.773119e+04
min,1.000000,0.000000e+00,1.000000,0.000000e+00,0.000000e+00,1.000000e+00,5.830000,0.000000e+00
25%,1.000000,3.801988e+04,1.000000,4.001204e+04,0.000000e+00,6.258600e+02,600.000000,2.032850e+03
50%,1.000000,9.841809e+04,1.000000,1.168401e+05,0.000000e+00,2.436740e+03,2000.000000,5.993380e+03
75%,1.000000,1.944364e+05,1.000000,2.181989e+05,5.757713e+04,1.189674e+04,3733.750000,1.125000e+04
max,90.000000,4.175205e+08,90.000000,2.055066e+07,1.769050e+07,7.150000e+06,411013.120000,1.027533e+06


### **1.3 Analizando columnas duplicadas**

Durante la vista al dataset se observa que aparentemente hay columnas que tienen los mismos datos, sera necesario verificarlos.

In [7]:
# ====== Verificar 3 columnas posiblemente iguales ('referencia', 'id_sub', 'referencia_det') =======
print('=== REFERENCIAS ===')
print(f'¿referencia == id_sub?')
print(f'Iguales     : {(df["referencia"] == df["id_sub"]).sum()}')
print(f'Distintos   : {(df["referencia"] != df["id_sub"]).sum()}')
print(f'Nulos id_sub: {df["id_sub"].isna().sum()}')

print("---------------------------------")
print(f'¿referencia == referencia_det?')
print(f'Iguales             : {(df["referencia"] == df["referencia_det"]).sum()}')
print(f'Distintos           : {(df["referencia"] != df["referencia_det"]).sum()}')
print(f'Nulos referencia_det: {df["referencia_det"].isna().sum()}')

print("---------------------------------")
# Ver ejemplos de los 3 juntos
print('Muestra de las 3 columnas:')
df[['referencia', 'id_sub', 'referencia_det']].head(5)

=== REFERENCIAS ===
¿referencia == id_sub?
Iguales     : 4726
Distintos   : 0
Nulos id_sub: 0
---------------------------------
¿referencia == referencia_det?
Iguales             : 4726
Distintos           : 0
Nulos referencia_det: 0
---------------------------------
Muestra de las 3 columnas:


,referencia,id_sub,referencia_det
0,SUB-JA-2024-231148,SUB-JA-2024-231148,SUB-JA-2024-231148
1,SUB-JA-2025-237883,SUB-JA-2025-237883,SUB-JA-2025-237883
2,SUB-JA-2025-242137,SUB-JA-2025-242137,SUB-JA-2025-242137
3,SUB-JA-2025-244592,SUB-JA-2025-244592,SUB-JA-2025-244592
4,SUB-JA-2025-245672,SUB-JA-2025-245672,SUB-JA-2025-245672


In [8]:
# ====== Verificar 2 columnas posiblemente iguales ('num_lotes_listado', 'lotes')=======

print('=== LOTES ===')
print(f'¿num_lotes_listado == lotes?')
iguales = (df['num_lotes_listado'] == df['lotes']).sum()
distintos = (df['num_lotes_listado'] != df['lotes']).sum()
print(f'Iguales   : {iguales}')
print(f'Distintos : {distintos}')

print("---------------------------------")
# Ver ejemplos de los 2 juntos
print('Muestra de las 2 columnas:')
df[['num_lotes_listado', 'lotes']].head(5)


=== LOTES ===
¿num_lotes_listado == lotes?
Iguales   : 4726
Distintos : 0
---------------------------------
Muestra de las 2 columnas:


,num_lotes_listado,lotes
0,1,1
1,1,1
2,4,4
3,2,2
4,1,1


In [9]:
# ====== Verificar las 2 columnas posiblemente iguales ('fecha_conclusion_listado', 'fecha_conclusion')=======

print('=== FECHA CONCLUSION ===')
print(f'fecha_conclusion_listado == fecha_conclusion?')
iguales = (df['fecha_conclusion_listado'] == df['fecha_conclusion']).sum()
distintos = (df['fecha_conclusion_listado'] != df['fecha_conclusion']).sum()
print(f'Iguales   : {iguales}')
print(f'Distintos : {distintos}')

print("---------------------------------")
# Ver ejemplos de los 2 juntos
print('Muestra de las 2 columnas:')
df[['fecha_conclusion_listado', 'fecha_conclusion']].head(7)



=== FECHA CONCLUSION ===
fecha_conclusion_listado == fecha_conclusion?
Iguales   : 0
Distintos : 4726
---------------------------------
Muestra de las 2 columnas:


,fecha_conclusion_listado,fecha_conclusion
0,01/10/2025 a las 18:00:00,2025-10-01
1,01/10/2025 a las 18:00:00,2025-10-01
2,01/10/2025 a las 18:00:00,2025-10-01
3,NaN,NaN
4,01/10/2025 a las 18:00:00,2025-10-01
5,01/10/2025 a las 18:00:00,2025-10-01
6,01/10/2025 a las 18:00:00,2025-10-01


### **1.4 Analizando valores nulos**

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4726 entries, 0 to 4725
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   referencia                4726 non-null   str    
 1   num_lotes_listado         4726 non-null   int64  
 2   organismo                 4726 non-null   str    
 3   expediente                3360 non-null   str    
 4   estado                    4726 non-null   str    
 5   fecha_conclusion_listado  4364 non-null   str    
 6   descripcion               4653 non-null   str    
 7   url_detalle               4726 non-null   str    
 8   id_sub                    4726 non-null   str    
 9   referencia_det            4726 non-null   str    
 10  tipo_subasta              4726 non-null   str    
 11  cuenta_expediente         3360 non-null   str    
 12  fecha_inicio              4364 non-null   str    
 13  fecha_conclusion          4364 non-null   str    
 14  cantidad_reclamada_

In [11]:
#============ Nulos por columna =========
nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos = nulos[nulos > 0].sort_values(ascending=False)

print('Columnas con valores nulos (%):')
print(nulos.to_string())

Columnas con valores nulos (%):
forma_adjudicacion          86.35
puja_minima_eur             72.05
expediente                  28.90
cuenta_expediente           28.90
tramo_pujas_eur             26.68
cantidad_reclamada_eur      22.24
valor_subasta_eur           14.05
tasacion_eur                13.65
deposito_eur                13.65
fecha_conclusion_listado     7.66
fecha_inicio                 7.66
fecha_conclusion             7.66
descripcion                  1.54
anuncio_boe                  0.06


**Explicación de decisión de los valores nulos**

| Columna | Nulos % | Decisión | Motivo |
|---------|---------|----------|--------|
| ***forma_adjudicacion*** | ***85%*** |  ~~Eliminar~~ | ***Solo aparece en subastas con varios lotes*** |
| puja_minima_eur | 78% |  Conservar | "Sin puja mínima" es información válida |
| tramo_pujas_eur | 29% |  Conservar | Útil para análisis económico |
| ***expediente*** | ***23%*** |  ~~Eliminar~~ | ***Solo judicial, no aporta al análisis*** |
| ***cuenta_expediente*** | ***23%*** |  ~~Eliminar~~ | ***Solo judicial, no aporta al análisis*** |
| cantidad_reclamada_eur | 16% |  Conservar | Útil para comparar judicial vs AEAT |
| valor_subasta_eur | 15% |  Conservar | Campo clave para hipótesis |
| tasacion_eur | 15% |  Conservar | Campo clave para hipótesis |
| deposito_eur | 15% |  Conservar | Útil para análisis económico |
| fecha_conclusion | 7% |  Conservar | Campo temporal importante |
| fecha_inicio | 7% |  Conservar | Campo temporal importante |
| descripcion | 3% |  Conservar | Descripción del bien |

### **1.5 Aplicando limpieza** 

* Durante la verificación de posibles columnas duplicadas, se encontraron **3** columnas duplicadas :

        'id_sub','referencia_det','num_lotes_listado'

* Se identificaron **2** columnas que no aportan al análisis:

        'url_detalle','anuncio_boe'

* Se identificaron **3** columnas con registros nulos que no aportan al análisis: 

        'forma_adjudicacion' - con 85% registros nulos  
        'expediente'         - 23% registros
        'cuenta_expediente'  - 23% registros

* Las fechas estaban en formato tipo 'String' y se decide convertir a Datetime


In [12]:
# Llamada a la funcion que elimina las 8 columnas
# Convierte fechas a datetime

df, cols_eliminadas = limpiar_dataframe(df)

In [13]:
#======= Verificando =====================
print(f'===== Limpieza aplicada =====')
print(f'Colummas eliminadas:\n{cols_eliminadas}')
print(f'Shape actual:{df.shape} ')
print(f'\n ===== Columnas restantes: =====')
df.info()
#for i, col in enumerate(df.columns):
#    print(f'  {i:2d}. {col}')


===== Limpieza aplicada =====
Colummas eliminadas:
['id_sub', 'referencia_det', 'num_lotes_listado', 'url_detalle', 'anuncio_boe', 'fecha_conclusion_listado', 'forma_adjudicacion', 'expediente', 'cuenta_expediente']
Shape actual:(4726, 14) 

 ===== Columnas restantes: =====
<class 'pandas.DataFrame'>
RangeIndex: 4726 entries, 0 to 4725
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   referencia              4726 non-null   str           
 1   organismo               4726 non-null   str           
 2   estado                  4726 non-null   str           
 3   descripcion             4653 non-null   str           
 4   tipo_subasta            4726 non-null   str           
 5   fecha_inicio            4364 non-null   datetime64[us]
 6   fecha_conclusion        4364 non-null   datetime64[us]
 7   cantidad_reclamada_eur  3675 non-null   float64       
 8   lotes                   

### **1.7 Calculando descuento_pct**
Mide cuánto por debajo de la tasación sale el precio de subasta.<br>

**- Cálculo:** Generalmente se calcula como:<br>
        **descuento_pct = (tasacion_eur - valor_subasta_eur) / tasacion_eur * 100**

**-Ejemplo:** <br>
tasacion_eur      = 100.000 €<br>
valor_subasta_eur =  60.000 €<br>
descuento_pct = (100.000 - 60.000) / 100.000 * 100 = 40%<br>

--> El inmueble sale a subasta con un 40% de descuento sobre su valor de mercado.




In [14]:
# Llamada a la función y asignar los resultados
df_limpio, mascara_calculo = calcular_pct(df)

In [15]:
# ===== Verificando ======
print(f'descuento_pct calculado')
print(f'Con descuento  : {mascara_calculo.sum()}')
print(f'Sin descuento  : {df["descuento_pct"].isna().sum()}')

descuento_pct calculado
Con descuento  : 1969
Sin descuento  : 2757


### **1.8 Calculando la duración de la subasta**

In [16]:
# Cuántos días duró cada subasta
calcular_duracion_dias(df)
print(f'==== duracion_dias calculado ====')

#para borrar
#print(f'\nEstadísticas:')
#print(df['duracion_dias'].describe())

==== duracion_dias calculado ====


### **1.9 Resumen del dataset**

In [17]:
resumen_dataset(df)

===== RESUMEN DEL DATASET =====
Filas   : 4,726
Columnas: 16

 Tipos de subasta:
tipo_subasta
JUDICIAL EN VÍA DE APREMIO    3147
AGENCIA TRIBUTARIA            1012
RECAUDACIÓN TRIBUTARIA         312
JUDICIAL VOLUNTARIA            149
JUDICIAL CONCURSAL              64
OTRAS SUBASTAS NOTARIALES       20
NOTARIAL HIPOTECARIA            15
ADMINISTRATIVA GENERAL           7

  Descuento sobre tasación (%):
count       1969.00
mean        -506.28
std        22616.44
min     -1003515.83
25%            0.00
50%            0.00
75%           17.37
max           99.00
Name: descuento_pct, dtype: float64
---------------------------


In [18]:
# Ver los casos extremos
print('=== VALORES EXTRAÑOS ===')
print('\nTop 5 descuentos más negativos:')
print(df[['referencia', 'tipo_subasta','tasacion_eur', 'valor_subasta_eur', 'descuento_pct']]
      .sort_values('descuento_pct')
      .head(5)
      .to_string())

print('\nTop 5 descuentos más altos:')
print(df[['referencia', 'tipo_subasta', 'tasacion_eur', 'valor_subasta_eur', 'descuento_pct']]
      .sort_values('descuento_pct', ascending=False)
      .head(5)
      .to_string())

print('\n¿Cuántos tienen tasacion_eur == 0?')
print(df[df['tasacion_eur'] == 0]['tasacion_eur'].count())

print('¿Cuántos tienen descuento negativo?')
print((df['descuento_pct'] < 0).sum())

print('¿Cuántos tienen descuento == 0?')
print((df['descuento_pct'] == 0).sum())

=== VALORES EXTRAÑOS ===

Top 5 descuentos más negativos:
              referencia                tipo_subasta  tasacion_eur  valor_subasta_eur  descuento_pct
4431  SUB-JA-2025-254826  JUDICIAL EN VÍA DE APREMIO        120.00         1204339.00    -1003515.83
3412  SUB-JV-2025-253747         JUDICIAL VOLUNTARIA     182794.64        18279464.00       -9900.00
3007  SUB-JA-2025-253412  JUDICIAL EN VÍA DE APREMIO      17821.40          178214.40        -900.00
3031  SUB-JA-2025-253510  JUDICIAL EN VÍA DE APREMIO      54018.01           93911.34         -73.85
1963  SUB-JA-2025-249929  JUDICIAL EN VÍA DE APREMIO      88919.57          120528.88         -35.55

Top 5 descuentos más altos:
                     referencia                tipo_subasta  tasacion_eur  valor_subasta_eur  descuento_pct
764          SUB-JA-2025-251488  JUDICIAL EN VÍA DE APREMIO    9910000.00           99100.00          99.00
4105  SUB-AT-2025-25R5086001073          AGENCIA TRIBUTARIA      21067.72             244.9

In [21]:
print('=== DIAGNÓSTICO TASACION ===')
print(f'tasacion_eur == 0: {(df["tasacion_eur"] == 0).sum()}')
print(f'tasacion_eur nula: {df["tasacion_eur"].isna().sum()}')
print(f'tasacion_eur > 0 : {(df["tasacion_eur"] > 0).sum()}')

print('\n=== DIAGNÓSTICO VALOR SUBASTA ===')
print(f'valor_subasta nulo: {df["valor_subasta_eur"].isna().sum()}')
print(f'valor_subasta == 0: {(df["valor_subasta_eur"] == 0).sum()}')
print(f'valor_subasta > 0 : {(df["valor_subasta_eur"] > 0).sum()}')

print('\n=== POR TIPO DE SUBASTA ===')
print(df.groupby('tipo_subasta')[['tasacion_eur', 'valor_subasta_eur']].agg(
    lambda x: x.isna().sum()
).rename(columns={'tasacion_eur': 'nulos_tasacion', 
                  'valor_subasta_eur': 'nulos_valor'}))

=== DIAGNÓSTICO TASACION ===
tasacion_eur == 0: 2093
tasacion_eur nula: 645
tasacion_eur > 0 : 1988

=== DIAGNÓSTICO VALOR SUBASTA ===
valor_subasta nulo: 664
valor_subasta == 0: 1
valor_subasta > 0 : 4061

=== POR TIPO DE SUBASTA ===
                            nulos_tasacion  nulos_valor
tipo_subasta                                           
ADMINISTRATIVA GENERAL                   0            0
AGENCIA TRIBUTARIA                       0            0
JUDICIAL CONCURSAL                      21           21
JUDICIAL EN VÍA DE APREMIO             548          548
JUDICIAL VOLUNTARIA                     21           21
NOTARIAL HIPOTECARIA                     3           12
OTRAS SUBASTAS NOTARIALES                4           14
RECAUDACIÓN TRIBUTARIA                  48           48


In [22]:
# ============= Volver a cargar dataset para verificar algunos datos en la web ====
df_verif = pd.concat([df_oct, df_nov, df_dic], ignore_index=True)
#df_verif.info()

# ======== Ver la URL de uno de estos registros en el df original =======
print(df_verif[df_verif['referencia'] == 'SUB-JA-2025-254826']['url_detalle'].values[0])
print(df_verif[df_verif['referencia'] == 'SUB-JV-2025-253747']['url_detalle'].values[0])
print(df_verif[df_verif['referencia'] == 'SUB-JA-2025-253412']['url_detalle'].values[0])

https://subastas.boe.es/detalleSubasta.php?idSub=SUB-JA-2025-254826&idBus=b1dsZHcyK2F3Z2VwRFVqaXQ1OGVPbGZZWnlDUzY0ekcvQTAzTDNNSks0RTJWL0h4UXRTRW1hdnlIRDJuMzJjMVgxYks0MUpxak5iWWpQNnZGaGhSMGVmUTNyUnpkS0JvWUtiSm56a1IxcjdYQXRpY2E2dHVFNnNtMXNoWDFvak1EeXhZVTZNQ3JCUlk0WVptemtmbHlLT21sbkF0TVpFOHRXYnp3Lzhhc0xzVmREWWNzVllPekhpTFp0NjRzQXJIaFdLY3h4WjNsc0tCR2JBOXN1d01lWXp0aG1iUjZTWTlmNkF3bHdGSFRRa2tCeG5WZnJFUzl2cmljMldsd1Q5VE1MWFJqVmF5VElCM2xud2s1UE5tckVPLzNHVjV1Vy8wbVpMdnpCRUdWeGxrUHFMODlmOVZVN09pVVEvcldOZGI,-1200-50
https://subastas.boe.es/detalleSubasta.php?idSub=SUB-JV-2025-253747&idBus=b1dsZHcyK2F3Z2VwRFVqaXQ1OGVPbGZZWnlDUzY0ekcvQTAzTDNNSks0RTJWL0h4UXRTRW1hdnlIRDJuMzJjMVgxYks0MUpxak5iWWpQNnZGaGhSMGVmUTNyUnpkS0JvWUtiSm56a1IxcjdYQXRpY2E2dHVFNnNtMXNoWDFvak1EeXhZVTZNQ3JCUlk0WVptemtmbHlLT21sbkF0TVpFOHRXYnp3Lzhhc0xzVmREWWNzVllPekhpTFp0NjRzQXJIaFdLY3h4WjNsc0tCR2JBOXN1d01lWXp0aG1iUjZTWTlmNkF3bHdGSFRRa2tCeG5WZnJFUzl2cmljMldsd1Q5VE1MWFJqVmF5VElCM2xud2s1UE5tckVPLzNHVjV1Vy8wbVpMdnpCRUdWeGxrUHFMODlmOVZVN09


### **1.9 Decisión sobre el dataset de análisis**

Durante la exploración se identificaron tres problemas en los datos:

**Problema 1 — Datos de origen incompletos**
Del totla 4,726 registros:
- 2,093 tienen ***tasacion_eur = 0***
- 645 tienen ***tasacion_eur** nula
- Solo 1,988 tienen tasación válida > 0

Verificando directamente en el portal del BOE se confirmó que estos valores provienen del origen los juzgados no siempre registran la tasación correctamente.

**Problema 2 — Anomalías en los datos**
Se detectaron registros donde ***valor_subasta_eur*** supera a ***tasacion_eur***, lo cual es económicamente incoherente. Verificando en el BOE se confirmó que son errores de origen:

- SUB-JA-2025-254826: Tasación = 120€, Valor subasta = 1.204.339€
- SUB-JA-2025-253747: Tasación = 182.794,64€, Valor subasta = 18.279.464.00€

**Problema 3 — Concentración de datos completos**
Al analizar nulos por tipo de subasta, se observó que los organismos tributarios ***(AEAT y Recaudación Tributaria)*** son los únicos con datos completos y coherentes:

| Tipo de subasta | Registros | Nulos tasación | Nulos valor subasta |
|----------------|-----------|---------------|-------------------|
| AGENCIA TRIBUTARIA | 1,012 | 0 | 0 |
| RECAUDACIÓN TRIBUTARIA | 312 | 48 | 48 |
| JUDICIAL EN VÍA DE APREMIO | 3,147 | 548 | 548 |
| JUDICIAL VOLUNTARIA | 149 | 21 | 21 |
| JUDICIAL CONCURSAL | 64 | 21 | 21 |

**Decisión:**
Se trabajará con los registros de organismos tributarios ***(AEAT y Recaudación Tributaria)*** con datos económicos completos y coherentes. Esto representa una muestra representativa y fiable para contrastar la hipótesis planteada.

### **1.10 Filtrar regitros con valores coherentes**
Condiciones para un registro válido:

1. Tipo de subasta (AGENCIA TRIBUTARIA, RECAUDACIÓN TRIBUTARIA)
2. tasacion_eur y valor_subasta_eur no nulos
3. tasacion_eur > valor_subasta_eur (la tasación debe ser mayor)

In [26]:
df_analisis = filtrar_tributarios(df)

print(f'Registros originales: {len(df)}')
print(f'Registros análisis  : {len(df_analisis)}')

Registros originales: 4726
Registros análisis  : 498


In [27]:
resumen_dataset(df_analisis)

===== RESUMEN DEL DATASET =====
Filas   : 498
Columnas: 16

 Tipos de subasta:
tipo_subasta
AGENCIA TRIBUTARIA        419
RECAUDACIÓN TRIBUTARIA     79

  Descuento sobre tasación (%):
count    498.00
mean      30.65
std       17.46
min        0.27
25%       20.00
50%       20.00
75%       40.00
max       98.84
Name: descuento_pct, dtype: float64
---------------------------


Se trabajará con los 498 registros de organismos tributarios **(419 de AEAT y 79 de Recaudación Tributaria)** con datos económicos completos y coherentes:
Esto representa el **10.5%** del dataset original pero es la muestra más fiable y representativa para contrastar la hipótesis.